In [54]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GAE
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from torch_geometric.utils import from_scipy_sparse_matrix
from torch_geometric.data import Data
import scipy.sparse as sp
import networkx as nx
import matplotlib.pyplot as plt
from torch.nn import BatchNorm1d
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch.nn import BatchNorm1d
from torch_geometric.utils import dropout_adj
import random

import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, BatchNorm
import numpy as np
import pandas as pd
import scipy.sparse as sp
from torch_geometric.data import Data
from torch_geometric.nn import GAE
from torch_geometric.utils import from_scipy_sparse_matrix


random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)


In [55]:
# Load the data
#data = pd.read_csv("/Users/nirwantandukar/input_data/nitrogen_medica/NORMALIZED_EXP/normalized_weight_matrix_nitrogen_roots_lowhigh_exp0.2filter.csv", index_col=0)
data = pd.read_csv("/Users/nirwantandukar/input_data/Database.DREAM4/NORMALIZED_EXP/normalized_weight_matrix_all_GRN_100_01_0.2filter.csv", index_col=0)
#data = pd.read_csv("/Users/nirwantandukar/input_data/PSU_P/NORMALIZED_EXP/normalized_weight_matrix_All_sig_genes_MaSigPro_NB_exp0.2filter.csv", index_col=0)
#data = pd.read_csv("/Users/nirwantandukar/input_data/GRN_R/adj_matrices_DREAM4_03_100_02filter_GRN_in_R_normalized.csv", index_col=0)
#data = pd.read_csv("/Users/nirwantandukar/input_data/e_coli/NORMALIZED_EXP/adj_matrices_ecoli_net3_02filter_GRN_in_R_normalized.csv", index_col=0)
print(f"Loaded data with shape: {data.shape}")

# Get the list of algorithms
columns = data.columns
algorithms = sorted(set(col.split('.')[0] for col in columns))
print(f"Algorithms found: {algorithms}")

num_genes = data.shape[0]
gene_names = data.index.tolist()
print(f"Number of genes: {num_genes}")

# Node features (identity matrix)
node_features = np.identity(num_genes)
print(f"Node features shape: {node_features.shape}")

Loaded data with shape: (100, 1700)
Algorithms found: ['GRN_ARACNE', 'GRN_BC3NET', 'GRN_C3NET', 'GRN_CLR', 'GRN_GENIE3_ET', 'GRN_GENIE3_RF', 'GRN_GRENIST', 'GRN_GRNBOOST2', 'GRN_INFERELATOR', 'GRN_KBOOST', 'GRN_LEAP', 'GRN_MEOMI', 'GRN_MRNET', 'GRN_MRNETB', 'GRN_NONLINEARODES', 'GRN_PCIT', 'GRN_TIGRESS']
Number of genes: 100
Node features shape: (100, 100)


In [56]:
# Functions
# Functions
class GCNEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers=3, dropout_rate=0.1, edge_dropout_rate=0.01):
    #def __init__(self, in_channels, hidden_channels, out_channels, num_layers=3, dropout_rate=0.1, edge_dropout_rate=0.1):
        super(GCNEncoder, self).__init__()
        self.num_layers = num_layers
        self.dropout_rate = dropout_rate
        self.edge_dropout_rate = edge_dropout_rate
        
        # First layer: input to 2x hidden size
        self.convs = torch.nn.ModuleList()
        self.convs.append(GCNConv(in_channels, 2 * hidden_channels, cached=False))
        
        # Add intermediate layers
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(2 * hidden_channels, 2 * hidden_channels, cached=False))
        
        # Final layer: hidden size to output size
        self.convs.append(GCNConv(2 * hidden_channels, out_channels, cached=False))
    
        # Batch normalization layers for each convolution layer
        #self.bns = torch.nn.ModuleList([BatchNorm1d(2 * hidden_channels) for _ in range(num_layers - 1)])
        self.bns = torch.nn.ModuleList([BatchNorm1d(2 * hidden_channels, momentum=0.1) for _ in range(num_layers - 1)])

    
        # Dropout rate
        self.dropout = torch.nn.Dropout(p=dropout_rate)
    
        # LeakyReLU instead of ReLU
        #self.activation = torch.nn.LeakyReLU(negative_slope=0.05)
        #self.activation = torch.nn.Sigmoid()
        #self.activation_hidden = torch.nn.LeakyReLU(negative_slope=0.05)  # For hidden layers
        self.activation_hidden = torch.nn.ReLU  # For hidden layers
        self.activation_output = torch.nn.Sigmoid()  # For final layer


    
    def forward(self, x, edge_index):
        # Apply edge dropout
        #edge_index, _ = dropout_adj(edge_index, p=self.edge_dropout_rate, training=self.training)
        
        # First layer (without batch normalization)
        x = self.convs[0](x, edge_index)
        #x = self.activation_hidden(x)
        x = F.relu(x)
    
    # Intermediate layers with batch normalization and dropout
        for i in range(1, self.num_layers - 1):
            x_residual = x # For residual connection
            x = self.convs[i](x, edge_index)
            x = self.bns[i-1](x) # Apply batch normalization
            #x = self.activation_hidden(x)
            x = F.relu(x)
            x = self.dropout(x)
            
            # Add residual connection (skip connection)
            x = x + x_residual
    
        # Final layer (no activation or batch norm)
        x = self.convs[-1](x, edge_index)
        x = F.relu(x)
        #x = self.activation_output(x)
        return x

#normalized_laplacian
def normalize_adj(adj):
    """Compute the normalized Laplacian."""
    adj = sp.coo_matrix(adj)
    rowsum = np.array(adj.sum(1)).flatten()
    d_inv_sqrt = np.power(rowsum + 1e-5, -0.5)
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.0
    d_mat_inv_sqrt = sp.diags(d_inv_sqrt)
    identity = sp.eye(adj.shape[0])  # Identity matrix of the same size as adj
    laplacian = identity - d_mat_inv_sqrt.dot(adj).dot(d_mat_inv_sqrt)  # Normalized Laplacian
    return laplacian.tocoo()


# GAE model with weight decay and dropout
def apply_gae_to_adjacency(adj_matrix, node_features, embedding_dim=10, epochs=200, dropout_rate=0.7, edge_dropout_rate=0.1, weight_decay=1e-4):
    # Convert adjacency matrix to sparse format
    adj_sparse = sp.csr_matrix(adj_matrix)
    # Normalize the adjacency matrix
    adj_normalized = normalize_adj(adj_sparse + sp.eye(adj_sparse.shape[0]))
    # Get edge_index and edge_weight
    edge_index, edge_weight = from_scipy_sparse_matrix(adj_normalized)
    # Create data object
    x = torch.tensor(node_features, dtype=torch.float)
    data = Data(x=x, edge_index=edge_index)

    # Define the GCNEncoder as the encoder for the GAE model
    encoder = GCNEncoder(in_channels=x.shape[1], hidden_channels=embedding_dim, out_channels=embedding_dim, dropout_rate=dropout_rate, edge_dropout_rate=edge_dropout_rate)
    
    # Instantiate the GAE model by passing the encoder
    model = GAE(encoder)

    # Training settings
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    data = data.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=weight_decay)

    # Training loop
    model.train()
    for epoch in range(1, epochs + 1):
        optimizer.zero_grad()
        z = model.encode(data.x, data.edge_index)
        loss = model.recon_loss(z, data.edge_index)
        loss.backward()
        optimizer.step()

    # After training, reconstruct adjacency matrix
    model.eval()
    with torch.no_grad():
        z = model.encode(data.x, data.edge_index)
        adj_reconstructed = model.decoder.forward_all(z).cpu().numpy()
        
    # Normalize the reconstructed adjacency matrix to be between 0 and 1
    adj_min = adj_reconstructed.min()
    adj_max = adj_reconstructed.max()
    normalized_adj = (adj_reconstructed - adj_min) / (adj_max - adj_min)
    
    return normalized_adj



# Convert ground truth matrix to edge list
def convert_matrix_to_edge_list(matrix):
    edges = []
    for source in matrix.index:
        for target in matrix.columns:
            if matrix.at[source, target] == 1:
                edges.append((source, target))
    return edges


# Define the function to perform the GAE process for each threshold and multiple iterations
def run_gae_multiple_thresholds(data, algorithms, thresholds, num_iterations=30):
    num_genes = data.shape[0]
    all_results = {}
    iteration_sums = {}

    for threshold in thresholds:
        print(f"Running for threshold: {threshold}")
        
        # Create a dictionary to store the results for each algorithm and threshold
        all_results[f'threshold_{threshold}'] = {}
        iteration_sums[f'threshold_{threshold}'] = {}

        # Initialize the iteration sum matrices for each threshold
        for iter_num in range(num_iterations):
            iteration_sums[f'threshold_{threshold}'][f'iter{iter_num + 1}_GCN_sum'] = np.zeros((num_genes, num_genes))

        for alg in algorithms:
            print(f"Running GAE for algorithm: {alg} with threshold: {threshold}")
            alg_columns = [col for col in data.columns if col.startswith(alg + '.')]
            alg_matrix = data[alg_columns].values

            # For each algorithm, create a list to store multiple runs
            all_results[f'threshold_{threshold}'][alg] = []

            for iteration in range(num_iterations):
                print(f"Iteration {iteration + 1}/{num_iterations} for algorithm: {alg}")
                
                # Apply GAE to the adjacency matrix of the algorithm
                adj_reconstructed = apply_gae_to_adjacency(alg_matrix, node_features)
                
                # Binarize the reconstructed adjacency matrix using the current threshold
                adj_binary = (adj_reconstructed > threshold).astype(int)

                # Store the binary adjacency matrix for this run
                all_results[f'threshold_{threshold}'][alg].append(adj_binary)

                # Add the binary matrix to the sum for this iteration and threshold
                iteration_sums[f'threshold_{threshold}'][f'iter{iteration + 1}_GCN_sum'] += adj_binary

    return all_results, iteration_sums


# Function to extract nodes and edges from the binary matrix
def count_nodes_edges(final_matrix):
    # Count edges in the upper triangle (to avoid double counting for undirected graphs)
    num_edges = np.sum(np.triu(final_matrix, k=1))
    
    # Count nodes (genes with at least one edge, i.e., rows/columns with non-zero values)
    connected_nodes = np.sum(np.any(final_matrix, axis=1))  # Count rows with non-zero values
    num_nodes = connected_nodes
    
    return num_nodes, num_edges


# Function to apply a second binarization threshold to the summed adjacency matrices
def apply_iteration_threshold(iteration_sums, thresholds, iter_thresholds):
    binary_sums = {}
    
    for threshold in thresholds:
        binary_sums[f'threshold_{threshold}'] = {}
        for iter_num in iteration_sums[f'threshold_{threshold}'].keys():
            binary_sums[f'threshold_{threshold}'][iter_num] = {}

            for iter_threshold in iter_thresholds:
                print(f"Applying iter_threshold {iter_threshold} to {iter_num} for threshold {threshold}")
                
                # Apply binarization on the summed adjacency matrix
                summed_matrix = iteration_sums[f'threshold_{threshold}'][iter_num]
                binary_matrix = (summed_matrix >= iter_threshold).astype(int)

                # Store the result
                binary_sums[f'threshold_{threshold}'][iter_num][f'iter_threshold_{iter_threshold}'] = binary_matrix
    
    return binary_sums


# Function to sum across iterations and apply a common threshold
def sum_across_iterations_and_binarize(binary_summed_matrices, thresholds, iter_thresholds, common_threshold):
    common_sums = {}

    for threshold in thresholds:
        common_sums[f'threshold_{threshold}'] = {}
        for iter_threshold in iter_thresholds:
            print(f"Summing across iterations for threshold {threshold} and iter_threshold {iter_threshold}")

            # Initialize the matrix for summing across iterations
            iteration_sum_matrix = np.zeros_like(binary_summed_matrices[f'threshold_{threshold}']['iter1_GCN_sum'][f'iter_threshold_{iter_threshold}'])

            # Sum the matrices across all iterations
            for iter_num in binary_summed_matrices[f'threshold_{threshold}'].keys():
                iteration_sum_matrix += binary_summed_matrices[f'threshold_{threshold}'][iter_num][f'iter_threshold_{iter_threshold}']

            # Apply the common threshold: if sum >= common_threshold, set 1; else set 0
            final_binarized_matrix = (iteration_sum_matrix >= common_threshold).astype(int)

            # Store the result
            common_sums[f'threshold_{threshold}'][f'iter_threshold_{iter_threshold}_common_threshold_{common_threshold}'] = final_binarized_matrix

    return common_sums



# Function to extract gene interactions (edges) from a binarized matrix
def extract_gene_interactions(final_matrix, gene_names):
    edges = []
    num_genes = final_matrix.shape[0]

    # Loop through the upper triangle of the matrix to avoid duplicates (since it's undirected)
    for i in range(num_genes):
        for j in range(i + 1, num_genes):
            if final_matrix[i, j] == 1:  # Check if there's an interaction
                edges.append(f"{gene_names[i]}-{gene_names[j]}")
    
    return edges


# Example to count nodes and edges for each threshold
def count_nodes_edges_for_thresholds(binary_summed_matrices, thresholds, iter_thresholds, common_threshold):
    node_edge_counts = {}

    for threshold in thresholds:
        for iter_threshold in iter_thresholds:
            final_matrix = binary_summed_matrices[f'threshold_{threshold}'][f'iter_threshold_{iter_threshold}_common_threshold_{common_threshold}']
            
            # Count nodes and edges
            num_nodes, num_edges = count_nodes_edges(final_matrix)
            
            # Store the counts
            node_edge_counts[f'threshold_{threshold}_iter_threshold_{iter_threshold}'] = {
                'Nodes': num_nodes,
                'Edges': num_edges
            }
    
    return node_edge_counts


In [163]:
# GAE model with weight decay and dropout
def apply_gae_to_adjacency(adj_matrix, node_features, embedding_dim=10, epochs=200, dropout_rate=0.7, edge_dropout_rate=0.1,weight_decay=1e-4):
    # Convert adjacency matrix to sparse format
    adj_sparse = sp.csr_matrix(adj_matrix)
    # Normalize the adjacency matrix
    adj_normalized = normalize_adj(adj_sparse + sp.eye(adj_sparse.shape[0]))
    # Get edge_index and edge_weight
    edge_index, edge_weight = from_scipy_sparse_matrix(adj_normalized)
    # Create data object
    x = torch.tensor(node_features, dtype=torch.float)
    data = Data(x=x, edge_index=edge_index)

    # Define the GCNEncoder as the encoder for the GAE model
    #encoder = GCNEncoder(in_channels=x.shape[1], hidden_channels=embedding_dim, out_channels=embedding_dim, dropout_rate=dropout_rate)
    encoder = GCNEncoder(in_channels=x.shape[1], hidden_channels=embedding_dim, out_channels=embedding_dim, dropout_rate=dropout_rate, edge_dropout_rate = edge_dropout_rate)
    #encoder = GCNEncoder(in_channels=x.shape[1], out_channels=embedding_dim, dropout_rate=dropout_rate)

    # Instantiate the GAE model by passing the encoder
    model = GAE(encoder)

    # Training settings
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    data = data.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=weight_decay)  # Applying weight decay le = 0.05

    # Training loop
    model.train()
    for epoch in range(1, epochs + 1):
        optimizer.zero_grad()
        z = model.encode(data.x, data.edge_index)
        loss = model.recon_loss(z, data.edge_index)
        loss.backward()
        optimizer.step()

    # After training, reconstruct adjacency matrix
    model.eval()
    with torch.no_grad():
        z = model.encode(data.x, data.edge_index)
        adj_reconstructed = model.decoder.forward_all(z).cpu().numpy()
        
    return adj_reconstructed

def apply_gae_to_adjacency(adj_matrix, node_features, embedding_dim=10, epochs=200, dropout_rate=0.7, edge_dropout_rate=0.01, weight_decay=1e-4, mc_samples=20):
    from sklearn.preprocessing import StandardScaler
    from tqdm import tqdm
    import torch.nn.utils

    # Normalize node features
    scaler = StandardScaler()
    node_features = scaler.fit_transform(node_features)

    # Convert adjacency matrix to sparse format and normalize
    adj_sparse = sp.csr_matrix(adj_matrix)
    adj_normalized = normalize_adj(adj_sparse + sp.eye(adj_sparse.shape[0]))
    edge_index, edge_weight = from_scipy_sparse_matrix(adj_normalized)

    # Create data object
    x = torch.tensor(node_features, dtype=torch.float)
    data = Data(x=x, edge_index=edge_index)

    # Define encoder and model
    encoder = GCNEncoder(in_channels=x.shape[1], hidden_channels=embedding_dim, out_channels=embedding_dim, dropout_rate=dropout_rate, edge_dropout_rate=edge_dropout_rate)
    model = GAE(encoder)

    # Device setup
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    data = data.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=weight_decay)

    # Training loop
    model.train()
    for epoch in tqdm(range(1, epochs + 1), desc="Training"):
        optimizer.zero_grad()
        z = model.encode(data.x, data.edge_index)
        loss = model.recon_loss(z, data.edge_index)
        loss.backward()
        optimizer.step()

    # Reconstruction with MC Sampling
    model.eval()
    reconstructed_adj_matrices = []
    with torch.no_grad():
        for _ in range(mc_samples):
            # Sample latent embeddings
            z = model.encode(data.x, data.edge_index)
            # Decode adjacency matrix
            adj_reconstructed = model.decoder.forward_all(z).cpu().numpy()
            reconstructed_adj_matrices.append(adj_reconstructed)
    
    # Aggregate the reconstructions (e.g., mean or median)
    reconstructed_adj_matrices = np.array(reconstructed_adj_matrices)
    avg_adj_reconstructed = reconstructed_adj_matrices.mean(axis=0)

    # Normalize aggregated adjacency matrix
    adj_min = avg_adj_reconstructed.min()
    adj_max = avg_adj_reconstructed.max()
    if adj_max == adj_min:
        normalized_adj = np.zeros_like(avg_adj_reconstructed)
    else:
        normalized_adj = (avg_adj_reconstructed - adj_min) / (adj_max - adj_min)

    return normalized_adj



In [181]:
# Define thresholds and number of iterations
thresholds = [0.6]  # Different thresholds to run
num_iterations = 100  # Number of runs for each threshold

# Define the additional binarization thresholds for the summed matrices
iter_thresholds = [1,2,3,4,5,6,7,8]  # Example thresholds for binarizing summed matrices

# Run the GAE process and store results for each threshold and algorithm
aggregated_results, iteration_sums = run_gae_multiple_thresholds(data, algorithms, thresholds, num_iterations)

# Apply the iteration-level thresholding
binary_summed_matrices = apply_iteration_threshold(iteration_sums, thresholds, iter_thresholds)

# Define the common threshold
common_threshold = 70  # Threshold to set edges based on consistency across iterations

# Sum across iterations and apply the common threshold
final_common_binarized_matrices = sum_across_iterations_and_binarize(binary_summed_matrices, thresholds, iter_thresholds, common_threshold)

# Assuming the gene names are the row/column names from your original data
gene_names = list(data.index)  # Replace 'data' with your actual DataFrame that contains the gene names

# Initialize a list to store the results for each threshold and iteration threshold
results = []


Running for threshold: 0.6
Running GAE for algorithm: GRN_ARACNE with threshold: 0.6
Iteration 1/100 for algorithm: GRN_ARACNE


Training: 100%|██████████████████████████████| 200/200 [00:00<00:00, 406.56it/s]


Iteration 2/100 for algorithm: GRN_ARACNE


Training: 100%|██████████████████████████████| 200/200 [00:00<00:00, 412.44it/s]


Iteration 3/100 for algorithm: GRN_ARACNE


Training: 100%|██████████████████████████████| 200/200 [00:00<00:00, 414.79it/s]


... (3,391 outputs left out)

Training: 100%|██████████████████████████████| 200/200 [00:00<00:00, 341.49it/s]


Iteration 100/100 for algorithm: GRN_TIGRESS


Training: 100%|██████████████████████████████| 200/200 [00:00<00:00, 350.27it/s]

Applying iter_threshold 1 to iter1_GCN_sum for threshold 0.6
Applying iter_threshold 2 to iter1_GCN_sum for threshold 0.6
Applying iter_threshold 3 to iter1_GCN_sum for threshold 0.6
Applying iter_threshold 4 to iter1_GCN_sum for threshold 0.6
Applying iter_threshold 5 to iter1_GCN_sum for threshold 0.6
Applying iter_threshold 6 to iter1_GCN_sum for threshold 0.6
Applying iter_threshold 7 to iter1_GCN_sum for threshold 0.6
Applying iter_threshold 8 to iter1_GCN_sum for threshold 0.6
Applying iter_threshold 1 to iter2_GCN_sum for threshold 0.6
Applying iter_threshold 2 to iter2_GCN_sum for threshold 0.6
Applying iter_threshold 3 to iter2_GCN_sum for threshold 0.6
Applying iter_threshold 4 to iter2_GCN_sum for threshold 0.6
Applying iter_threshold 5 to iter2_GCN_sum for threshold 0.6
Applying iter_threshold 6 to iter2_GCN_sum for threshold 0.6
Applying iter_threshold 7 to iter2_GCN_sum for threshold 0.6
Applying iter_threshold 8 to iter2_GCN_sum for threshold 0.6
Applying iter_threshold 

In [182]:
# For each threshold, iter_threshold, and final binarized matrix, calculate the number of edges, nodes, and extract interactions
for threshold in thresholds:
    for iter_threshold in iter_thresholds:
        # Access the final binarized matrix for this combination of threshold and iter_threshold
        final_matrix = final_common_binarized_matrices[f'threshold_{threshold}'][f'iter_threshold_{iter_threshold}_common_threshold_{common_threshold}']
        
        # Count nodes and edges
        num_nodes, num_edges = count_nodes_edges(final_matrix)
        
        # Extract gene interactions (edges)
        predicted_edges = extract_gene_interactions(final_matrix, gene_names)
        
        # Store the result in the list
        results.append({
            'Iterations': num_iterations,
            'Common_Iterations': common_threshold,
            'Iter_Threshold': iter_threshold,
            'Individual_Threshold': threshold,
            'Nodes': num_nodes,
            'Edges': num_edges,
            'Gene_Interactions': ','.join(predicted_edges)  # Store gene interactions in the format G1-G2,G3-G4,...
        })

# Convert the results to a DataFrame for better visualization
results_df = pd.DataFrame(results)
results_df


,Iterations,Common_Iterations,Iter_Threshold,Individual_Threshold,Nodes,Edges,Gene_Interactions
0,100,70,1,0.6,98,976,"G1-G16,G1-G18,G1-G22,G1-G23,G1-G24,G1-G25,G1-G..."
1,100,70,2,0.6,84,568,"G1-G29,G1-G31,G1-G48,G1-G57,G1-G59,G1-G60,G1-G..."
2,100,70,3,0.6,75,362,"G1-G69,G10-G14,G10-G43,G10-G44,G10-G64,G10-G94..."
3,100,70,4,0.6,62,245,"G10-G14,G10-G43,G10-G44,G10-G64,G14-G44,G16-G5..."
4,100,70,5,0.6,47,172,"G10-G43,G10-G44,G18-G22,G18-G46,G18-G47,G18-G4..."
5,100,70,6,0.6,39,119,"G10-G43,G10-G44,G18-G22,G18-G46,G18-G47,G18-G4..."
6,100,70,7,0.6,32,67,"G18-G22,G18-G53,G18-G57,G22-G46,G22-G47,G22-G4..."
7,100,70,8,0.6,23,40,"G18-G53,G18-G57,G22-G46,G22-G53,G22-G57,G23-G2..."


In [183]:
# Display the final results DataFrame with edges, nodes, and gene interactions
print(results_df)

# Save the final DataFrame (if needed in the future)
#results_df.to_csv('Ecoli_net3_GCN_10080.csv', index=False)


   Iterations  Common_Iterations  Iter_Threshold  Individual_Threshold  Nodes  \
0         100                 70               1                   0.6     98   
1         100                 70               2                   0.6     84   
2         100                 70               3                   0.6     75   
3         100                 70               4                   0.6     62   
4         100                 70               5                   0.6     47   
5         100                 70               6                   0.6     39   
6         100                 70               7                   0.6     32   
7         100                 70               8                   0.6     23   

   Edges                                  Gene_Interactions  
0    976  G1-G16,G1-G18,G1-G22,G1-G23,G1-G24,G1-G25,G1-G...  
1    568  G1-G29,G1-G31,G1-G48,G1-G57,G1-G59,G1-G60,G1-G...  
2    362  G1-G69,G10-G14,G10-G43,G10-G44,G10-G64,G10-G94...  
3    245  G10-G14,G10-

In [184]:
# Step 1: Load known (ground truth) interactions
file_path = "/Users/nirwantandukar/input_data/Database.DREAM4/GS/dream4_100_01_gs.csv"
ground_truth = pd.read_csv(file_path, index_col=0)

# Convert ground truth matrix to edge list
def convert_matrix_to_edge_list(matrix):
    edges = []
    for source in matrix.index:
        for target in matrix.columns:
            if matrix.at[source, target] == 1:
                edges.append((source, target))
    return edges

# Get the ground truth edges
known_edges = set(convert_matrix_to_edge_list(ground_truth))

# Step 2: Update results_df to include true positives, false positives, and false negatives for each row
true_positive_counts = []
false_positive_counts = []
false_negative_counts = []

for index, row in results_df.iterrows():
    if pd.notna(row['Gene_Interactions']):
        predicted_edges = [tuple(edge.split('-')) for edge in row['Gene_Interactions'].split(',')]
        predicted_edges_set = set(predicted_edges)

        # Calculate true positives, false positives, and false negatives
        true_positives = predicted_edges_set.intersection(known_edges)
        false_positives = predicted_edges_set.difference(known_edges)
        false_negatives = known_edges.difference(predicted_edges_set)

        # Append counts to lists
        true_positive_counts.append(len(true_positives))
        false_positive_counts.append(len(false_positives))
        false_negative_counts.append(len(false_negatives))
    else:
        true_positive_counts.append(0)
        false_positive_counts.append(0)
        false_negative_counts.append(0)

# Add the counts to the DataFrame
results_df['True_Positives'] = true_positive_counts
results_df['False_Positives'] = false_positive_counts
results_df['False_Negatives'] = false_negative_counts

# Print updated results_df
print(results_df)


   Iterations  Common_Iterations  Iter_Threshold  Individual_Threshold  Nodes  \
0         100                 70               1                   0.6     98   
1         100                 70               2                   0.6     84   
2         100                 70               3                   0.6     75   
3         100                 70               4                   0.6     62   
4         100                 70               5                   0.6     47   
5         100                 70               6                   0.6     39   
6         100                 70               7                   0.6     32   
7         100                 70               8                   0.6     23   

   Edges                                  Gene_Interactions  True_Positives  \
0    976  G1-G16,G1-G18,G1-G22,G1-G23,G1-G24,G1-G25,G1-G...              62   
1    568  G1-G29,G1-G31,G1-G48,G1-G57,G1-G59,G1-G60,G1-G...              44   
2    362  G1-G69,G10-G14,G10-G43,

In [103]:
results_df

,Iterations,Common_Iterations,Iter_Threshold,Individual_Threshold,Nodes,Edges,Gene_Interactions,True_Positives,False_Positives,False_Negatives
0,10,8,1,0.6,100,1334,"G1-G10,G1-G14,G1-G16,G1-G18,G1-G22,G1-G23,G1-G...",68,1266,107
1,10,8,2,0.6,100,709,"G1-G16,G1-G18,G1-G22,G1-G23,G1-G29,G1-G31,G1-G...",55,654,120
2,10,8,3,0.6,99,476,"G1-G16,G1-G23,G1-G29,G1-G31,G1-G36,G1-G57,G1-G...",43,433,132
3,10,8,4,0.6,87,352,"G1-G29,G1-G59,G1-G69,G1-G72,G10-G14,G10-G43,G1...",38,314,137
4,10,8,5,0.6,77,268,"G1-G69,G10-G14,G10-G43,G10-G44,G10-G64,G11-G15...",34,234,141
5,10,8,6,0.6,68,205,"G10-G14,G10-G43,G10-G44,G14-G43,G14-G44,G15-G5...",28,177,147
6,10,8,7,0.6,57,160,"G10-G43,G10-G44,G18-G22,G18-G46,G18-G47,G18-G4...",20,140,155
7,10,8,8,0.6,45,114,"G10-G43,G10-G44,G18-G22,G18-G46,G18-G47,G18-G4...",15,99,160


In [154]:
# Save the final DataFrame (if needed in the future)
#results_df.to_csv('GRN_R_GCN_100_1_02filter_2020_normalized2.csv', index=False)


In [3]:
# Step 1: Compute Graph Convolution Operation (GCN Layer)

# Compute A~ = A + I (Adjacency Matrix with self-loops)
identity_matrix = np.eye(adj_matrix_example.shape[0])
adj_tilde = adj_matrix_example + identity_matrix

# Compute Degree Matrix D~
degree_tilde = np.diag(adj_tilde.sum(axis=1))

# Compute Normalized Graph Laplacian D~^(-1/2) * A~ * D~^(-1/2)
D_inv_sqrt = np.linalg.inv(np.sqrt(degree_tilde))
graph_convolution_matrix = D_inv_sqrt @ adj_tilde @ D_inv_sqrt

# Step 2: Compute Latent Embeddings (Z) using Eigenvectors
eigenvalues, eigenvectors = np.linalg.eigh(graph_convolution_matrix)
latent_embeddings_example = eigenvectors[:, 1:4]  # Selecting top 3 non-trivial embeddings

# Step 3: Compute Reconstructed Adjacency Matrix (Decoder)
adj_reconstructed_example = np.dot(latent_embeddings_example, latent_embeddings_example.T)
adj_reconstructed_example = (adj_reconstructed_example - np.min(adj_reconstructed_example)) / (
    np.max(adj_reconstructed_example) - np.min(adj_reconstructed_example)
)  # Normalize

# Step 4: Compute Binary Cross-Entropy Loss for Reconstruction
epsilon = 1e-9  # To avoid log(0)
loss_matrix = -(
    adj_matrix_example * np.log(adj_reconstructed_example + epsilon) +
    (1 - adj_matrix_example) * np.log(1 - adj_reconstructed_example + epsilon)
)
loss_value = np.sum(loss_matrix)  # Total loss

# Step 5: Graph Dropout Mechanism (Randomly Drop Edges with Probability p=0.5)
dropout_prob = 0.5
dropout_mask = np.random.rand(*adj_matrix_example.shape) > dropout_prob
adj_dropout = adj_matrix_example * dropout_mask

# Step 6: Evaluation Metrics (AUC and AUPRC)
from sklearn.metrics import roc_auc_score, average_precision_score

true_edges = adj_matrix_example.flatten()
pred_edges = adj_reconstructed_example.flatten()

# Compute ROC-AUC and AUPRC
auc_score = roc_auc_score(true_edges, pred_edges)
auprc_score = average_precision_score(true_edges, pred_edges)

# Display matrices and results
tools.display_dataframe_to_user(name="Graph Convolution (GCN) Matrix", dataframe=pd.DataFrame(graph_convolution_matrix))
tools.display_dataframe_to_user(name="Latent Embeddings (Z)", dataframe=pd.DataFrame(latent_embeddings_example))
tools.display_dataframe_to_user(name="Reconstructed Adjacency Matrix", dataframe=pd.DataFrame(adj_reconstructed_example))
tools.display_dataframe_to_user(name="Loss Matrix", dataframe=pd.DataFrame(loss_matrix))
tools.display_dataframe_to_user(name="Graph Dropout Matrix", dataframe=pd.DataFrame(adj_dropout))
tools.display_dataframe_to_user(name="Evaluation Metrics", dataframe=pd.DataFrame({"Metric": ["ROC-AUC", "AUPRC"], "Value": [auc_score, auprc_score]}))


NameError: name 'adj_matrix_example' is not defined